In [1]:
# Task 5: Content-Based TED Talks Recommendation Engine

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Load the cleaned TED Talks dataset
df = pd.read_csv("../data/processed/ted_talks_cleaned.csv")

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

Dataset loaded successfully.
Dataset shape: (4005, 24)


In [4]:
# Generate TF-IDF vectors
vectorizer = TfidfVectorizer(max_features=10000)
tfidf_matrix = vectorizer.fit_transform(df["combined_text"])

print("TF-IDF vectorization completed.")
print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF vectorization completed.
TF-IDF matrix shape: (4005, 10000)


In [5]:
# Calculate cosine similarity matrix
similarity_matrix = cosine_similarity(tfidf_matrix)

print("Cosine similarity matrix calculated.")
print("Similarity matrix shape:", similarity_matrix.shape)

Cosine similarity matrix calculated.
Similarity matrix shape: (4005, 4005)


In [6]:

# Create a reusable recommendation function
def recommend_talks(title, top_n=5):

    matching_titles = df["title"].str.strip().str.lower()
    search_title = title.strip().lower()

    if search_title not in matching_titles.values:
        raise ValueError("TED Talk title not found.")

    talk_index = matching_titles[matching_titles == search_title].index[0]

    similarity_scores = similarity_matrix[talk_index]

    similar_indices = similarity_scores.argsort()[::-1]

    recommendations = []

    for index in similar_indices:
        if index != talk_index:
            recommendations.append({
                "TED Talk Title": df.iloc[index]["title"],
                "Similarity Score": round(similarity_scores[index], 4)
            })

        if len(recommendations) == top_n:
            break

    return pd.DataFrame(recommendations)


In [10]:
# Test with different TED Talk titles
recommend_talks("The best stats you've ever seen")

,TED Talk Title,Similarity Score
0,Let my dataset change your mindset,0.8581
1,New insights on poverty,0.8154
2,How not to be ignorant about the world,0.7293
3,The good news of the decade? We're winning the...,0.7264
4,Asia's rise -- how and when,0.7238


In [11]:
# Test with another TED Talk title
recommend_talks("Simplicity sells")

,TED Talk Title,Similarity Score
0,Design and discovery,0.7515
1,"5 predictions, from 1984",0.7367
2,Smart failure for a fast-changing world,0.7362
3,Cool tricks your phone can do,0.7248
4,Play! Experiment! Discover!,0.7239


In [12]:
# Test with a TED Talk title
try:
    recommendations = recommend_talks("Artificial Intelligence and Future Technology")
    display(recommendations)

except ValueError as e:
    print(e)

TED Talk title not found.


## Observations

1. The recommendation function successfully returns the top five similar TED Talks for different input titles.
2. The selected TED Talk is excluded from its own recommendations.
3. Higher cosine similarity scores indicate greater textual similarity between TED Talks.
4. Invalid TED Talk titles are handled using ValueError.
5. TF-IDF vectorization and cosine similarity are calculated only once, making the recommendation function reusable and efficient.